# Importing necessary Libraries

In [8]:
import pandas as pd

# Step 1: Data Understanding & Preprocessing

## Loading Data

In [9]:
df = pd.read_excel("./Data/Leave.xlsx")

In [10]:
df

,SERVICE NO,DIV,DEP,LOC,LEAVE DATE,LEAVE TYPE,NO DAYS
0,M.G.R.LIVERA,DYA,SMT,PP4,2024-01-03,AL,1.0
1,M.G.R.LIVERA,DYA,SMT,PP4,2024-01-06,AL,0.5
2,M.G.R.LIVERA,DYA,SMT,PP4,2024-05-20,AL,1.0
3,M.G.R.LIVERA,DYA,SMT,PP4,2024-04-03,AL,1.0
4,M.G.R.LIVERA,DYA,SMT,PP4,2024-06-15,AL,0.5
...,...,...,...,...,...,...,...
57635,K.P.A.DULANJANA,DPR,SDE,MAS,2024-11-26,SK,1.0
57636,H.M.Y.G.V.S. SENAVIRATHNA,DYA,SMT,MAT,2024-11-14,DL,1.0
57637,R.A.S.DHARMAWANSHA,DPR,SEA,ELW,2024-12-28,SK,0.5
57638,D.M.P.MADHUSANKA,DMD,SRC,FIU,2024-12-30,SK,0.5


## Handle missing values

In [11]:
df.isnull().sum()

SERVICE NO       0
DIV              0
DEP              0
LOC              0
LEAVE DATE       0
LEAVE TYPE    1111
NO DAYS          0
dtype: int64

In [12]:
df = df.drop(columns=['LEAVE TYPE'])

In [13]:
df

,SERVICE NO,DIV,DEP,LOC,LEAVE DATE,NO DAYS
0,M.G.R.LIVERA,DYA,SMT,PP4,2024-01-03,1.0
1,M.G.R.LIVERA,DYA,SMT,PP4,2024-01-06,0.5
2,M.G.R.LIVERA,DYA,SMT,PP4,2024-05-20,1.0
3,M.G.R.LIVERA,DYA,SMT,PP4,2024-04-03,1.0
4,M.G.R.LIVERA,DYA,SMT,PP4,2024-06-15,0.5
...,...,...,...,...,...,...
57635,K.P.A.DULANJANA,DPR,SDE,MAS,2024-11-26,1.0
57636,H.M.Y.G.V.S. SENAVIRATHNA,DYA,SMT,MAT,2024-11-14,1.0
57637,R.A.S.DHARMAWANSHA,DPR,SEA,ELW,2024-12-28,0.5
57638,D.M.P.MADHUSANKA,DMD,SRC,FIU,2024-12-30,0.5


In [14]:
df.isnull().sum()

SERVICE NO    0
DIV           0
DEP           0
LOC           0
LEAVE DATE    0
NO DAYS       0
dtype: int64

In [15]:
df.nunique()

SERVICE NO    1529
DIV             13
DEP             40
LOC             92
LEAVE DATE     365
NO DAYS          2
dtype: int64

## Feature Engineering

### Converting "LEAVE DATE" to datatime format

In [16]:
df['LEAVE DATE'] = pd.to_datetime(df['LEAVE DATE'], errors='coerce')

In [17]:
df.describe()

,LEAVE DATE,NO DAYS
count,57640,57640.000000
mean,2024-07-15 07:33:05.121443328,0.828001
min,2023-07-06 00:00:00,0.500000
25%,2024-04-10 00:00:00,0.500000
50%,2024-07-22 00:00:00,1.000000
75%,2024-10-19 00:00:00,1.000000
max,2025-01-30 00:00:00,1.000000
std,NaN,0.237522


### Extract Month, Day of the Week, and Year

In [21]:
df['LEAVE MONTH'] = df['LEAVE DATE'].dt.month
df['LEAVE YEAR'] = df['LEAVE DATE'].dt.year
df['LEAVE DAY OF WEEK'] = df['LEAVE DATE'].dt.dayofweek

```python
df['LEAVE DATE'].dt.month
```
- Extracts the month from the LEAVE DATE column.
- Returns the month as a number (e.g., 1 for January, 2 for February, etc.). 

```python
df['LEAVE DATE'].dt.dayofweek
```
- Extracts the day of the week from the LEAVE DATE column.
- Returns the day of the week as a number, where:
    - 0 = Monday
    - 1 = Tuesday
    - 2 = Wednesday
    - 3 = Thursday
    - 4 = Friday
    - 5 = Saturday
    - 6 = Sunday

In [22]:
df

,SERVICE NO,DIV,DEP,LOC,LEAVE DATE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK
0,M.G.R.LIVERA,DYA,SMT,PP4,2024-01-03,1.0,1,2024,2
1,M.G.R.LIVERA,DYA,SMT,PP4,2024-01-06,0.5,1,2024,5
2,M.G.R.LIVERA,DYA,SMT,PP4,2024-05-20,1.0,5,2024,0
3,M.G.R.LIVERA,DYA,SMT,PP4,2024-04-03,1.0,4,2024,2
4,M.G.R.LIVERA,DYA,SMT,PP4,2024-06-15,0.5,6,2024,5
...,...,...,...,...,...,...,...,...,...
57635,K.P.A.DULANJANA,DPR,SDE,MAS,2024-11-26,1.0,11,2024,1
57636,H.M.Y.G.V.S. SENAVIRATHNA,DYA,SMT,MAT,2024-11-14,1.0,11,2024,3
57637,R.A.S.DHARMAWANSHA,DPR,SEA,ELW,2024-12-28,0.5,12,2024,5
57638,D.M.P.MADHUSANKA,DMD,SRC,FIU,2024-12-30,0.5,12,2024,0


### Calculate Time Gap Between Leaves

In [23]:
# Sort data by employee and leave date
df = df.sort_values(by=['SERVICE NO', 'LEAVE DATE'])

In [ ]:
# Compute difference between consecutive leaves per employee
df['PREV LEAVE DATE'] = df.groupby('SERVICE NO')['LEAVE DATE'].shift(1)
df['DAYS SINCE LAST LEAVE'] = (df['LEAVE DATE'] - df['PREV LEAVE DATE']).dt.days